In [1]:
import re
from collections import defaultdict

In [2]:
try:
    from nltk.corpus import stopwords
    import nltk; nltk.download('stopwords', quiet=True)
    STOP = set(stopwords.words('english'))
except Exception:                      # fallback if NLTK / data is unavailable
    STOP = set("i me my we our you your he him his she her it its they them their what which who "
               "this that these those am is are was were be been being have has had do does did a an "
               "the and but if or because as until while of at by for with about against between into "
               "through during before after above below to from up down in out on off over under again "
               "further then once here there when where why how all any both each few more most other "
               "some such no nor not only own same so than too very can will just don should now".split())

In [3]:
with open('file.txt', 'w', encoding='utf8') as f:
    f.write("This is the first word.\nThis is the second text, Hello! How are you?\nThis is the third, this is it now.")

In [4]:
with open('file.txt', encoding='utf8') as f:
    lines = f.read().split('\n')
print("Number of lines in file is:", len(lines))
print(lines)

def tokenize(line):
    line = re.sub(r"[^\w\s]", " ", line.lower())   # remove punctuation, lowercase
    return [w for w in line.split() if w not in STOP]

Number of lines in file is: 3
['This is the first word.', 'This is the second text, Hello! How are you?', 'This is the third, this is it now.']


In [5]:
# --- build word-level inverted index: word -> [line numbers] ---
inverted = defaultdict(list)
for no, line in enumerate(lines, start=1):
    for w in tokenize(line):
        if no not in inverted[w]:
            inverted[w].append(no)

print("\nInverted index:")
print(dict(inverted))


Inverted index:
{'first': [1], 'word': [1], 'second': [2], 'text': [2], 'hello': [2], 'third': [3]}


In [6]:
# --- retrieval ---
def search(query):
    terms = tokenize(query)
    if not terms: return []
    result = set(inverted.get(terms[0], []))
    for t in terms[1:]:                      # AND query
        result &= set(inverted.get(t, []))
    return sorted(result)

for q in ["hello", "second text", "third", "missing"]:
    docs = search(q)
    print(f"\nQuery: '{q}' -> lines {docs}")
    for d in docs:
        print("   ", lines[d-1])


Query: 'hello' -> lines [2]
    This is the second text, Hello! How are you?

Query: 'second text' -> lines [2]
    This is the second text, Hello! How are you?

Query: 'third' -> lines [3]
    This is the third, this is it now.

Query: 'missing' -> lines []
